# Exploratory only

Source of truth = Python modules under `app/` and `scripts/`. These notebooks are retained for EDA, experiment notes, and demo walkthroughs only.

# Full Image Inference with Tiling and Global Fusion

## Objective

The final model was trained on tiled aerial images in order to improve small-object detection.

However, real-world aerial imagery is typically available as full-size images, not tiles.

This notebook demonstrates how to apply the trained tiled model to a full image by:

- splitting the image into overlapping tiles
- running inference on each tile
- projecting detections back to global image coordinates
- merging predictions into a final full-image result

This step bridges model training and real-world deployment.

##Imports and Model Loading

In [ ]:
import os
import cv2
import glob
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from ultralytics import YOLO

# chemins
ZIP_PATH = "/content/inference_test .zip"
EXTRACT_PATH = "/content/inference_test"
MODEL_PATH = "/content/best_tiled.pt"

# extraire les images
os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Images extraites")

# lister les images
image_paths = glob.glob("/content/inference_test/**/*.png", recursive=True) + \
              glob.glob("/content/inference_test/**/*.jpg", recursive=True) + \
              glob.glob("/content/inference_test/**/*.jpeg", recursive=True)

print("Nombre d'images trouvées :", len(image_paths))
print(image_paths[:5])

# charger le modèle
model = YOLO(MODEL_PATH)
print("Modèle chargé avec succès")

##Load Full Aerial Image

In [ ]:
# choisir une image complète
FULL_IMG_PATH = image_paths[0]
print("Image choisie :", FULL_IMG_PATH)

# charger l'image
full_img = cv2.imread(FULL_IMG_PATH)

if full_img is None:
    raise ValueError(f"Impossible de charger l'image : {FULL_IMG_PATH}")

# conversion BGR -> RGB pour affichage
full_img_rgb = cv2.cvtColor(full_img, cv2.COLOR_BGR2RGB)

print("Shape de l'image :", full_img.shape)

# afficher
plt.figure(figsize=(12, 12))
plt.imshow(full_img_rgb)
plt.axis("off")
plt.show()

##Generate Overlapping Tiles

In [ ]:
from app.inference import OBBDetector
from app.utils import compute_iou_aabb, generate_tiles, nms_obb


def generate_inference_tiles(img, tile_size=1024, overlap=200):
    for x1, y1, x2, y2 in generate_tiles(img.shape[1], img.shape[0], tile_size, overlap):
        yield img[y1:y2, x1:x2].copy(), x1, y1


def polygon_to_bbox(poly):
    xs = [point[0] for point in poly]
    ys = [point[1] for point in poly]
    return min(xs), min(ys), max(xs), max(ys)


def compute_iou(box1, box2):
    return compute_iou_aabb(box1, box2)


def global_nms(predictions, iou_threshold=0.5):
    return nms_obb(predictions, iou_threshold)


In [ ]:
tiles = generate_inference_tiles(full_img, TILE_SIZE, OVERLAP)

print("Nombre de tiles générées :", len(tiles))
print("Exemple tile shape :", tiles[0]["image"].shape)

##Run Inference on Each Tile

In [ ]:
all_predictions = []

for i, tile_data in enumerate(tiles):

    tile_img = tile_data["image"]

    results = model.predict(
        source=tile_img,
        conf=0.3,
        device=0,
        verbose=False
    )

    result = results[0]

    if result.obb is None:
        continue

    obb = result.obb

    for j in range(len(obb.cls)):
        cls = int(obb.cls[j].item())
        conf = float(obb.conf[j].item())
        xyxyxyxy = obb.xyxyxyxy[j].cpu().numpy()

        all_predictions.append({
            "cls": cls,
            "conf": conf,
            "polygon": xyxyxyxy,
            "x_offset": tile_data["x_offset"],
            "y_offset": tile_data["y_offset"]
        })

print("Nombre total de détections :", len(all_predictions))

##Reproject Detections to Global Coordinates

In [ ]:
global_predictions = []

for pred in all_predictions:

    poly = pred["polygon"].copy()

    # appliquer offset
    poly[:, 0] += pred["x_offset"]
    poly[:, 1] += pred["y_offset"]

    global_predictions.append({
        "cls": pred["cls"],
        "conf": pred["conf"],
        "polygon": poly
    })

print("Détections globales :", len(global_predictions))

##Merge Predictions with Global NMS

In [ ]:
from app.inference import OBBDetector
from app.utils import compute_iou_aabb, generate_tiles, nms_obb


def generate_inference_tiles(img, tile_size=1024, overlap=200):
    for x1, y1, x2, y2 in generate_tiles(img.shape[1], img.shape[0], tile_size, overlap):
        yield img[y1:y2, x1:x2].copy(), x1, y1


def polygon_to_bbox(poly):
    xs = [point[0] for point in poly]
    ys = [point[1] for point in poly]
    return min(xs), min(ys), max(xs), max(ys)


def compute_iou(box1, box2):
    return compute_iou_aabb(box1, box2)


def global_nms(predictions, iou_threshold=0.5):
    return nms_obb(predictions, iou_threshold)


##Visualize Final Full-Image Predictions

In [ ]:
import cv2
import matplotlib.pyplot as plt

# copie de l'image originale
vis_img = full_img_rgb.copy()

# mapping des classes (optionnel mais propre)
CLASS_NAMES = {
    0: "plane",
    1: "ship",
    2: "small-vehicle",
    3: "large-vehicle"
}

for pred in final_predictions:

    poly = pred["polygon"].astype(int)
    cls = pred["cls"]
    conf = pred["conf"]

    # dessiner le polygone
    cv2.polylines(
        vis_img,
        [poly],
        isClosed=True,
        color=(0, 255, 0),
        thickness=2
    )

    # texte
    label = f"{CLASS_NAMES.get(cls, cls)} {conf:.2f}"

    x, y = poly[0]
    cv2.putText(
        vis_img,
        label,
        (x, y - 5),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (255, 0, 0),
        1,
        cv2.LINE_AA
    )

# afficher
plt.figure(figsize=(12, 12))
plt.imshow(vis_img)
plt.axis("off")
plt.show()

In [ ]:
vis_img = full_img_rgb.copy()

COLORS = {
    0: (255, 0, 0),   # plane
    1: (0, 255, 255), # ship
    2: (0, 255, 0),
    3: (255, 255, 0)
}

for pred in final_predictions:

    poly = pred["polygon"].astype(int)
    cls = pred["cls"]

    conf = float(pred["conf"])
    if conf > 1:
        conf = conf / 100

    color = COLORS.get(cls, (255, 255, 255))

    thickness = 3 if conf > 0.6 else 1

    cv2.polylines(vis_img, [poly], True, color, thickness)

    label = f"{CLASS_NAMES.get(cls, cls)} {conf:.2f}"

    x, y = poly[0]

    cv2.putText(
        vis_img,
        label,
        (x, y - 5),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        color,
        2
    )

plt.figure(figsize=(12, 12))
plt.imshow(vis_img)
plt.axis("off")
plt.show()

## Conclusion

This notebook demonstrates how to bridge the gap between model training and real-world deployment for aerial object detection.

### Key Achievements

* Successfully applied a model trained on tiled images to full-resolution aerial images
* Implemented a complete inference pipeline:

  * Image tiling with overlap
  * Tile-level prediction
  * Projection to global coordinates
  * Prediction merging using global NMS
* Achieved consistent detection performance on full images despite training constraints

### Key Insight

Models trained on tiled data cannot be directly applied to full images without adaptation.

Applying the same tiling strategy at inference time ensures:

* Consistent object scale
* Improved detection of small objects
* Better alignment with training conditions

### Technical Takeaway

The combination of tiling + global fusion acts as a scalable inference strategy for high-resolution imagery.

This approach:

* Reduces memory constraints
* Maintains detection accuracy
* Enables processing of arbitrarily large images

### Next Steps

* Package this pipeline into a reusable inference module
* Deploy as a FastAPI service with a `/predict` endpoint
* Optimize inference speed (batching, parallel tiles)
* Explore advanced merging strategies (soft-NMS, weighted fusion)

### Final Thought

In computer vision systems, performance is not only determined by the model architecture, but also by how data is structured and processed at inference time.

A well-designed inference pipeline is essential for translating model performance into real-world impact.
